In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [1]:
import sys, subprocess, shutil
from pathlib import Path

# 正确项目根目录
src = Path("/kaggle/input/datasets/roxyjiang12180318/ultralytics-cbam-project")
code_dir = Path("/kaggle/working/ultralytics-cbam-project")

# 强制复制最新版源码，避免沿用上一次 working 里的旧副本
if code_dir.exists():
    shutil.rmtree(code_dir)
shutil.copytree(src, code_dir)

# 生成 AMP check 需要的占位图
import numpy as np, cv2
assets_dir = code_dir / "ultralytics" / "assets"
assets_dir.mkdir(parents=True, exist_ok=True)
cv2.imwrite(str(assets_dir / "bus.jpg"), np.ones((640, 640, 3), dtype=np.uint8) * 128)

# 用当前 notebook 的同一个 Python 安装
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-e", str(code_dir), "-q"],
    check=True,
)

# 确保导入的是我们的源码
if str(code_dir) not in sys.path:
    sys.path.insert(0, str(code_dir))

import ultralytics
print("Ultralytics path:", ultralytics.__file__)

from ultralytics import YOLO

yaml_path = code_dir / "ultralytics" / "cfg" / "models" / "11" / "yolo11-no-c2psa-cbam.yaml"
print("YAML exists:", yaml_path.exists())

model = YOLO(str(yaml_path))

results = model.train(
    data="/kaggle/input/datasets/roxyjiang12180318/maizeleaf/AAAA/data.yaml",
    epochs=200,
    imgsz=640,
    batch=16,
    device=0,
    workers=4,
    optimizer="AdamW",
    lr0=0.001,
    lrf=0.01,
    patience=30,
    save=True,
    save_period=10,
    project="/kaggle/working/no-c2psa-cbam",
    name="exp",
    exist_ok=True,
    amp=True,
    cos_lr=True,
    warmup_epochs=3,
    pretrained=False,
)

print("Best model:", results.save_dir / "weights" / "best.pt")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics path: /kaggle/working/ultralytics-cbam-project/ultralytics/__init__.py
YAML exists: True
WARNING ⚠️ no model scale passed. Assuming scale='n'.
New https://pypi.org/project/ultralytics/8.4.115 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.89 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/kaggle/input/datasets/roxyjiang12180318/maizeleaf/AAA

In [2]:
import shutil
from pathlib import Path
from IPython.display import FileLink, display

exp_dir = Path("/kaggle/working/no-c2psa-cbam/exp")
zip_path = Path("/kaggle/working/no-c2psa-cbam_exp_results.zip")

print("结果目录存在:", exp_dir.exists())
print("包含文件:")
for f in sorted(exp_dir.rglob("*")):
    if f.is_file():
        print(f"  {f.relative_to(exp_dir)}  ({f.stat().st_size / 1024:.1f} KB)")

if zip_path.exists():
    zip_path.unlink()
shutil.make_archive(str(zip_path.with_suffix("")), "zip", str(exp_dir))

print("\n打包完成:", zip_path)
print("大小:", round(zip_path.stat().st_size / 1024 / 1024, 2), "MB")

# 点击下载整个结果包
display(FileLink(str(zip_path)))

# 如果 zip 下载不顺利，单独下载 results.csv 备用
display(FileLink(str(exp_dir / "results.csv")))

结果目录存在: True
包含文件:
  BoxF1_curve.png  (161.0 KB)
  BoxPR_curve.png  (155.8 KB)
  BoxP_curve.png  (196.7 KB)
  BoxR_curve.png  (154.8 KB)
  args.yaml  (1.7 KB)
  confusion_matrix.png  (125.8 KB)
  confusion_matrix_normalized.png  (140.7 KB)
  labels.jpg  (181.4 KB)
  results.csv  (17.0 KB)
  results.png  (269.2 KB)
  train_batch0.jpg  (557.9 KB)
  train_batch1.jpg  (511.0 KB)
  train_batch2.jpg  (557.7 KB)
  val_batch0_labels.jpg  (528.8 KB)
  val_batch0_pred.jpg  (523.1 KB)
  val_batch1_labels.jpg  (570.5 KB)
  val_batch1_pred.jpg  (561.5 KB)
  val_batch2_labels.jpg  (530.2 KB)
  val_batch2_pred.jpg  (538.8 KB)
  weights/best.pt  (5635.9 KB)
  weights/epoch0.pt  (21907.8 KB)
  weights/epoch10.pt  (21908.7 KB)
  weights/epoch100.pt  (21920.0 KB)
  weights/epoch110.pt  (21921.2 KB)
  weights/epoch120.pt  (21922.5 KB)
  weights/epoch130.pt  (21923.7 KB)
  weights/epoch20.pt  (21910.0 KB)
  weights/epoch30.pt  (21911.2 KB)
  weights/epoch40.pt  (21912.5 KB)
  weights/epoch50.pt  (21913.7 K

/kaggle/working/no-c2psa-cbam_exp_results.zip

/kaggle/working/no-c2psa-cbam/exp/results.csv